In [1]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from dotenv import load_dotenv
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
from transformers import (
    Trainer, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
)

load_dotenv()

True

In [2]:
df = pd.read_csv("../../data/dontpatronizeme_pcl.tsv", sep="\t")
df["label_pcl"] = (df["label"] >= 2).astype(int)
len(df)

10469

In [3]:
def prepend(row):
    text = str(row["text"]).strip()
    return f"Keyword: {row["keyword"]}. Country: {row["country_code"]}. Text: {text}"


# Apply the function across the rows (axis=1)
df["cleaned_text"] = df.apply(prepend, axis=1)
df.head()

,par_id,art_id,keyword,country_code,text,label,label_pcl,cleaned_text
0,1,@@24942188,hopeless,ph,"We 're living in times of absolute insanity , ...",0,0,Keyword: hopeless. Country: ph. Text: We 're l...
1,2,@@21968160,migrant,gh,"In Libya today , there are countless number of...",0,0,Keyword: migrant. Country: gh. Text: In Libya ...
2,3,@@16584954,immigrant,ie,White House press secretary Sean Spicer said t...,0,0,Keyword: immigrant. Country: ie. Text: White H...
3,4,@@7811231,disabled,nz,Council customers only signs would be displaye...,0,0,Keyword: disabled. Country: nz. Text: Council ...
4,5,@@1494111,refugee,ca,""" Just like we received migrants fleeing El Sa...",0,0,"Keyword: refugee. Country: ca. Text: "" Just li..."


In [4]:
df_train_dev_labels = pd.read_csv("../../data/train_semeval_parids-labels.csv")
df_test_labels = pd.read_csv("../../data/dev_semeval_parids-labels.csv")
(
    len(df_train_dev_labels),
    len(df_test_labels),
    len(df_train_dev_labels) + len(df_test_labels)
)

(8375, 2094, 10469)

In [5]:
df_train_dev = df[df["par_id"].isin(df_train_dev_labels["par_id"])]
len(df_train_dev)

8375

In [6]:
df_test = df[df["par_id"].isin(df_test_labels["par_id"])]
len(df_test)

2094

In [7]:
df_train, df_dev = train_test_split(
    df_train_dev,
    test_size=0.1,
    random_state=42,
    shuffle=True,
    stratify=df_train_dev["label_pcl"],
)
len(df_train), len(df_dev)

(7537, 838)

In [8]:
# Oversampling
pcl_pos = df_train[df_train["label_pcl"] == 1]
df_train = pd.concat([df_train, pcl_pos, pcl_pos], ignore_index=True).sample(frac=1, random_state=42)
len(df_train)

8967

In [9]:
df_train["label_pcl"].value_counts(normalize=True)

label_pcl
0    0.76079
1    0.23921
Name: proportion, dtype: float64

In [10]:
df_dev["label_pcl"].value_counts(normalize=True)

label_pcl
0    0.905728
1    0.094272
Name: proportion, dtype: float64

In [11]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")


def tokenize_function(examples):
    return tokenizer(examples["cleaned_text"], padding="max_length", truncation=True, max_length=256)

In [12]:
hf_dataset_train = Dataset.from_pandas(df_train)
tokenized_dataset_train = hf_dataset_train.map(tokenize_function, batched=True)
tokenized_dataset_train = tokenized_dataset_train.rename_column("label_pcl", "labels")
tokenized_dataset_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/8967 [00:00<?, ? examples/s]

In [13]:
hf_dataset_dev = Dataset.from_pandas(df_dev)
tokenized_dataset_dev = hf_dataset_dev.map(tokenize_function, batched=True)
tokenized_dataset_dev = tokenized_dataset_dev.rename_column("label_pcl", "labels")
tokenized_dataset_dev.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/838 [00:00<?, ? examples/s]

In [14]:
# Check for Apple Silicon (mps), then NVIDIA (cuda), then fall back to CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
device

device(type='mps')

In [15]:
class_weights = compute_class_weight(
    "balanced",
    classes=np.unique(df_train["label_pcl"]),
    y=df_train["label_pcl"]
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)


# 2. Create a Custom Trainer to use the weights
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.logits
        # Apply the custom weights to the Cross Entropy Loss
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


class_weights_tensor

tensor([0.6572, 2.0902], device='mps:0')

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, preds, pos_label=1)
    return {"f1_pcl": f1}

In [17]:
model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

training_args = TrainingArguments(
    output_dir="",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pcl",
    greater_is_better=True,
    logging_steps=50,
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset_train,
    eval_dataset=tokenized_dataset_dev,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/Users/henrytsyu/Imperial/70

Epoch,Training Loss,Validation Loss,F1 Pcl
1,0.405684,0.425358,0.514286
2,0.222245,0.573926,0.529730
3,0.073756,0.959781,0.536585
4,0.038461,1.052452,0.527607
5,0.026351,1.539905,0.431655
6,0.000098,1.389756,0.513158
7,0.014105,1.544962,0.521127
8,0.000050,1.474732,0.520548


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=4488, training_loss=0.11816624554164447, metrics={'train_runtime': 2575.0638, 'train_samples_per_second': 27.858, 'train_steps_per_second': 1.743, 'total_flos': 9437267333652480.0, 'train_loss': 0.11816624554164447, 'epoch': 8.0})

In [18]:
# 1. Get the model's predictions on your internal dev set
predictions = trainer.predict(tokenized_dataset_dev)

# 2. Convert raw logits into 0 or 1 labels
preds = np.argmax(predictions.predictions, axis=-1)

# 3. Get the true labels
labels = predictions.label_ids

# 4. Calculate the F1 score for the positive class (PCL = 1)
f1 = f1_score(labels, preds)
print("Internal Dev F1 Score:", f1)

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Internal Dev F1 Score: 0.5365853658536586


In [19]:
# This will show you exactly how many False Positives vs False Negatives you have
print(classification_report(labels, preds))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95       759
           1       0.52      0.56      0.54        79

    accuracy                           0.91       838
   macro avg       0.74      0.75      0.74       838
weighted avg       0.91      0.91      0.91       838



In [20]:
trainer.save_model("./BestModel")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [21]:
tokenizer.save_pretrained("./BestModel")

('./BestModel/tokenizer_config.json', './BestModel/tokenizer.json')

In [22]:
# Create dev.txt

hf_dataset_test = Dataset.from_pandas(df_test)
tokenized_dataset_test = hf_dataset_test.map(tokenize_function, batched=True)
tokenized_dataset_test = tokenized_dataset_test.rename_column("label_pcl", "labels")
tokenized_dataset_test.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 1. Get the model's predictions on test set
predictions = trainer.predict(tokenized_dataset_test)

# 2. Convert raw logits into 0 or 1 labels
preds = np.argmax(predictions.predictions, axis=-1)

# 3. Get the true labels
labels = predictions.label_ids

# 4. Calculate the F1 score for the positive class (PCL = 1)
f1 = f1_score(labels, preds)
print("Test F1 Score:", f1)

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Test F1 Score: 0.5995085995085995


In [23]:
print(classification_report(labels, preds))

              precision    recall  f1-score   support

           0       0.96      0.95      0.96      1895
           1       0.59      0.61      0.60       199

    accuracy                           0.92      2094
   macro avg       0.77      0.78      0.78      2094
weighted avg       0.92      0.92      0.92      2094

